In [2]:
# RUN THIS CELL FIRST TO IMPORT ALL NECESSARY LIBRARIES
import pandas as pd
import numpy as np
import requests
import time
import io
import os

### EXTRACT DATA

In [ ]:
# RUN THIS CELL ONLY TO UPDATE THE ISTAT DATA
# this cell save the istat data in a raw data folder in order to avoid running the api call every time
url = "https://esploradati.istat.it/SDMXWS/rest/data/41_983"
header = {"Accept":"application/vnd.sdmx.data+csv;version=1.0.0"}
response = requests.get(url, headers=header)
print(response.status_code)

with open("data/raw/istat_incidenti.csv", "w", encoding="utf-8") as f:
    f.write(response.text)

200


In [30]:
# RUN THIS CELL ONCE TO DOWNLOAD SITUAS DATA
# this cell save the situas data from 31-12-2001 to 31-12-2024 in 24 different csv files
date_ranges = [f"{year}-12-31" for year in range(2001, 2025)]

# to download files in csv format is necessary to set json and header parameters.
# You can acknowledge this by using dev tools on the web page at the below specified url.
json_situas = {"orderFields":[],"orderDirects":[],"pFilterFields":[],"pFilterValues":[]}
header_situas = {"Content-Type": "application/json-patch+json"}

for i in date_ranges:
    url_situas = f"https://situas.istat.it/ShibO2Module/api/Report/Spool/{i}/74?&pdoctype=CSV"
    time.sleep(1)
    response_situas = requests.post(url_situas, json = json_situas, headers = header_situas)
    print(response_situas.status_code)
    if response_situas.status_code == 200:
        with open(f"data/raw/situas_{i[0:4]}.csv", "w", encoding="utf-8") as f:
            f.write(response_situas.text)
    else:
        # if an http status code different from 200 is encounterd, this make sure that cell execution is stopped
        raise SystemExit(f"Abort due to http error {response_situas.status_code}")
print(f"All done - {len(date_ranges)} files downloaded")


200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
All done - 24 files downloaded


### CLEAN DATA

We begin cleaning data with `SITUAS data` csv file

In [3]:
situas_df = pd.read_csv("data/raw/situas_2001.csv", sep = ";", decimal = ",")
situas_df["Anno"] = 2001
print(f"Year 2001:{situas_df.shape}")
situas_df = [situas_df]

for i in range(2002,2025):
    tmp_df = pd.read_csv(f"data/raw/situas_{i}.csv", sep = ";", decimal = ",")
    tmp_df["Anno"] = i # add a column with the year of the data

# Analyzing the csv situas data, it can be noticed that from 2015 on there is another column 
# not present in the previous years "Codice Provincia (Storico)"". This is simply the code that 
# in the previous years is called "Codice Provincia/Uts" which from 2015 on contains different values. 
# So, we can drop "Codice Provincia/Uts" and change "Codice Provincia (Storico)"" name in "Codice Provincia/Uts"
# in order to obtain a uniform dataframe.
    if i in range(2015,2025):
        tmp_df.drop(columns=["Codice Provincia/Uts"], inplace = True)
        tmp_df.rename(columns={"Codice Provincia (Storico)": "Codice Provincia/Uts"}, inplace = True)

    # check if columns name are the same among csv files in both directions
    check_column_names_sx = set(situas_df[0].columns) - set(tmp_df.columns)
    check_column_names_dx = set(tmp_df.columns) - set(situas_df[0].columns)

    # check if columns order are the same among csv files
    check_column_order = list(situas_df[0].columns) == list(tmp_df.columns)

    if len(check_column_names_sx) == 0 and len(check_column_names_dx) == 0 and check_column_order:
        print(f"Year {i}: {tmp_df.shape}")
        situas_df.append(tmp_df)
    else:
        print(situas_df[0].columns)
        print(tmp_df.columns)
        raise SystemExit(f"Abort due to column name mismatch in situas_{i}.csv")
situas_df = pd.concat(situas_df, ignore_index = True)



Year 2001:(8102, 17)
Year 2002: (8102, 17)
Year 2003: (8100, 17)
Year 2004: (8101, 17)
Year 2005: (8101, 17)
Year 2006: (8101, 17)
Year 2007: (8101, 17)
Year 2008: (8101, 17)
Year 2009: (8100, 17)
Year 2010: (8094, 17)
Year 2011: (8092, 17)
Year 2012: (8092, 17)
Year 2013: (8090, 17)
Year 2014: (8057, 17)
Year 2015: (8046, 17)
Year 2016: (7998, 17)
Year 2017: (7978, 17)
Year 2018: (7954, 17)
Year 2019: (7914, 17)
Year 2020: (7903, 17)
Year 2021: (7904, 17)
Year 2022: (7904, 17)
Year 2023: (7900, 17)
Year 2024: (7896, 17)


From the shapes of the previous cell, we can notice that the number of rows change for every situas csv file. Since that every record in these csv files is a different town, this means that the number of town changes every year. This happens because some towns splits or merge toghether with others. This kind of town usually are very small so it's difficult that they could be interesting for investment from business. So we can simply keep just the towns that are present in every csv file. In order to confirm this strategy, we can calculate how many towns are in common between all csv files.

In [4]:
# Determine the total number of distinct years covered by the dataset.
tot_years = situas_df["Anno"].nunique()

# Count the number of years in which each municipality appears.
count_years_per_town = situas_df.groupby("Codice Comune (alfanumerico)")["Anno"].nunique()

# Select towns that are present in all years of the dataset.
towns_common_code = count_years_per_town[count_years_per_town == tot_years].index
print(len(towns_common_code))

7488


Since that the majority of towns don't change through years, we can assess that the bigger towns are not excluded from the analysis and we can confirm the strategy. We drop changing towns because to do a correct analysis, we have to work on towns that have the same amount of data 

In [5]:
# Keep only towns that are present in every year of the dataset,
# then reset the DataFrame index after filtering.
situas_df = situas_df[
    situas_df['Codice Comune (alfanumerico)'].isin(towns_common_code)
]
situas_df.reset_index(inplace=True, drop=True)
print(situas_df.info())
situas_df.head(10)

<class 'pandas.DataFrame'>
RangeIndex: 179712 entries, 0 to 179711
Data columns (total 17 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   ï»¿Codice Ripartizione geografica  179712 non-null  int64  
 1   Codice Regione                     179712 non-null  int64  
 2   Codice Provincia/Uts               179712 non-null  int64  
 3   Codice Comune (alfanumerico)       179712 non-null  int64  
 4   Codice Comune (numerico)           179712 non-null  int64  
 5   Comune                             179688 non-null  str    
 6   Comune (dizione straniera)         2904 non-null    str    
 7   Sigla automobilistica              177504 non-null  str    
 8   Capoluogo di Provincia/Uts         179712 non-null  int64  
 9   Capoluogo di Regione               179712 non-null  int64  
 10  Popolazione legale                 179712 non-null  float64
 11  Anno Censimento                    179712 non-null

,ï»¿Codice Ripartizione geografica,Codice Regione,Codice Provincia/Uts,Codice Comune (alfanumerico),Codice Comune (numerico),Comune,Comune (dizione straniera),Sigla automobilistica,Capoluogo di Provincia/Uts,Capoluogo di Regione,Popolazione legale,Anno Censimento,Superficie (Kmq),Anno (Superficie),Popolazione residente,Anno (Popolazione residente),Anno
0,1,1,1,1001,1001,AgliÃ¨,NaN,TO,0,0,2574.0,2001,"13,1462",2001,2557,2001,2001
1,1,1,1,1002,1002,Airasca,NaN,TO,0,0,3554.0,2001,"15,7393",2001,3543,2001,2001
2,1,1,1,1003,1003,Ala di Stura,NaN,TO,0,0,479.0,2001,"46,3315",2001,480,2001,2001
3,1,1,1,1004,1004,Albiano d'Ivrea,NaN,TO,0,0,1696.0,2001,"11,7315",2001,1687,2001,2001
4,1,1,1,1006,1006,Almese,NaN,TO,0,0,5658.0,2001,"17,8756",2001,5648,2001,2001
5,1,1,1,1007,1007,Alpette,NaN,TO,0,0,300.0,2001,"5,6261",2001,293,2001,2001
6,1,1,1,1008,1008,Alpignano,NaN,TO,0,0,16648.0,2001,"11,9193",2001,16647,2001,2001
7,1,1,1,1009,1009,Andezeno,NaN,TO,0,0,1705.0,2001,"7,4861",2001,1712,2001,2001
8,1,1,1,1010,1010,Andrate,NaN,TO,0,0,476.0,2001,"9,3085",2001,474,2001,2001
9,1,1,1,1011,1011,Angrogna,NaN,TO,0,0,777.0,2001,"38,8782",2001,777,2001,2001


In [6]:
# drop the column "Comune (dizione straniera) since that have a lot nan values and is not useful for our analysis"
situas_df.drop(columns=['Comune (dizione straniera)'],inplace=True)

In [7]:
# Loading the situas data, it can be noticed that the first column name is not correct. 
# It is called "ï»¿Codice Ripartizione geografica" instead of "Codice Ripartizione geografica". 
# This is due to a problem with the encoding of the csv file. We can fix this by renaming the column.
situas_df.rename(columns={'ï»¿Codice Ripartizione geografica': 'Codice Ripartizione geografica'}, inplace=True)
print(situas_df.info())
print(situas_df.sample(10))

<class 'pandas.DataFrame'>
RangeIndex: 179712 entries, 0 to 179711
Data columns (total 16 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   Codice Ripartizione geografica  179712 non-null  int64  
 1   Codice Regione                  179712 non-null  int64  
 2   Codice Provincia/Uts            179712 non-null  int64  
 3   Codice Comune (alfanumerico)    179712 non-null  int64  
 4   Codice Comune (numerico)        179712 non-null  int64  
 5   Comune                          179688 non-null  str    
 6   Sigla automobilistica           177504 non-null  str    
 7   Capoluogo di Provincia/Uts      179712 non-null  int64  
 8   Capoluogo di Regione            179712 non-null  int64  
 9   Popolazione legale              179712 non-null  float64
 10  Anno Censimento                 179712 non-null  int64  
 11  Superficie (Kmq)                179712 non-null  str    
 12  Anno (Superficie)          

From the previous cell we can see there are some nan values (about 2000) in the column "Sigla automobilistica".

In [8]:
print(situas_df[pd.isna(situas_df['Sigla automobilistica'])])
print(situas_df['Codice Regione'][pd.isna(situas_df['Sigla automobilistica'])].unique())

        Codice Ripartizione geografica  Codice Regione  Codice Provincia/Uts  \
4944                                 4              15                    63   
4945                                 4              15                    63   
4946                                 4              15                    63   
4947                                 4              15                    63   
4948                                 4              15                    63   
...                                ...             ...                   ...   
177255                               4              15                    63   
177256                               4              15                    63   
177257                               4              15                    63   
177258                               4              15                    63   
177259                               4              15                    63   

        Codice Comune (alfanumerico)  C

Investigating on this columns, we can see that all nan values have "Codice Regione" $= 15$. This code stands for "Campania". We can fill these nan with the correct code using the column "Codice Provincia"

In [9]:
# we can discover the province code for each of the 5 province of Campania.
# Then we can fill nan values in column "Sigla automobilistica" with the correct string code.
province_code = []
province_name = []
for i in range(0,len(situas_df)):
    if situas_df['Comune'].iloc[i] in ['Napoli','Caserta','Salerno','Avellino','Benevento']:
        province_code.append(situas_df['Codice Provincia/Uts'].iloc[i])
        province_name.append(situas_df['Comune'].iloc[i])
province_campania_df = pd.DataFrame({'Codice Provincia/Uts': province_code, 'Nome Provincia/Uts': province_name})
print(province_campania_df)

for i in range(0,len(situas_df)):
    if pd.isna(situas_df['Sigla automobilistica'].iloc[i]):
        if situas_df['Codice Provincia/Uts'].iloc[i] == 61:
            situas_df.loc[i,'Sigla automobilistica'] = 'CE'
        elif situas_df['Codice Provincia/Uts'].iloc[i] == 62:
            situas_df.loc[i,'Sigla automobilistica'] = 'BN'
        elif situas_df['Codice Provincia/Uts'].iloc[i] == 63:
            situas_df.loc[i,'Sigla automobilistica']  = 'NA'
        elif situas_df['Codice Provincia/Uts'].iloc[i] == 64:
            situas_df.loc[i,'Sigla automobilistica'] = 'AV'
        elif situas_df['Codice Provincia/Uts'].iloc[i] == 65:
            situas_df.loc[i,'Sigla automobilistica'] = 'SA'

print(situas_df.info())
print(situas_df.sample(10))


     Codice Provincia/Uts Nome Provincia/Uts
0                      61            Caserta
1                      62          Benevento
2                      63             Napoli
3                      64           Avellino
4                      65            Salerno
..                    ...                ...
115                    61            Caserta
116                    62          Benevento
117                    63             Napoli
118                    64           Avellino
119                    65            Salerno

[120 rows x 2 columns]
<class 'pandas.DataFrame'>
RangeIndex: 179712 entries, 0 to 179711
Data columns (total 16 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   Codice Ripartizione geografica  179712 non-null  int64  
 1   Codice Regione                  179712 non-null  int64  
 2   Codice Provincia/Uts            179712 non-null  int64  
 3   Codice Comune (alfanu

In [ ]:
# Save situas dataframe to csv file in data clean folder
situas_df.to_csv("data/clean/situas_df.csv", index=False, sep = ";", decimal = ",")

Now, we can move to clean `ISTAT data` csv file

In [11]:
istat_df = pd.read_csv("data/raw/istat_incidenti.csv")
print(istat_df.info())
print(istat_df.sample(10))

<class 'pandas.DataFrame'>
RangeIndex: 573552 entries, 0 to 573551
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   DATAFLOW          573552 non-null  str    
 1   FREQ              573552 non-null  str    
 2   REF_AREA          573552 non-null  int64  
 3   DATA_TYPE         573552 non-null  str    
 4   RESULT            573552 non-null  str    
 5   TIME_PERIOD       573552 non-null  int64  
 6   OBS_VALUE         573552 non-null  int64  
 7   OBS_STATUS        0 non-null       float64
 8   NOTE_DS           0 non-null       float64
 9   NOTE_REF_AREA     0 non-null       float64
 10  NOTE_DATA_TYPE    0 non-null       float64
 11  NOTE_RESULT       0 non-null       float64
 12  NOTE_TIME_PERIOD  0 non-null       float64
 13  BASE_PER          0 non-null       float64
 14  UNIT_MEAS         0 non-null       float64
 15  UNIT_MULT         0 non-null       float64
dtypes: float64(9), int64(3), str(4)

Since the columns from "OBS_STATUS" to "UNIT_MULT" contain only nan values, and these columns are meant to contain only optional data like metadata or notes, we can drop them

In [12]:
# we cycle over all columns of the istat_df dataframe and check if there are any missing values. 
# If there are, we drop the column.
for i in istat_df.columns:
    if pd.isna(istat_df[i]).sum() > 0:
        print(f"Column {i} has {pd.isna(istat_df[i]).sum()} missing values")
        istat_df.drop(columns=[i],inplace=True)
print(istat_df.info())
print(istat_df.sample(10))

Column OBS_STATUS has 573552 missing values
Column NOTE_DS has 573552 missing values
Column NOTE_REF_AREA has 573552 missing values
Column NOTE_DATA_TYPE has 573552 missing values
Column NOTE_RESULT has 573552 missing values
Column NOTE_TIME_PERIOD has 573552 missing values
Column BASE_PER has 573552 missing values
Column UNIT_MEAS has 573552 missing values
Column UNIT_MULT has 573552 missing values
<class 'pandas.DataFrame'>
RangeIndex: 573552 entries, 0 to 573551
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   DATAFLOW     573552 non-null  str  
 1   FREQ         573552 non-null  str  
 2   REF_AREA     573552 non-null  int64
 3   DATA_TYPE    573552 non-null  str  
 4   RESULT       573552 non-null  str  
 5   TIME_PERIOD  573552 non-null  int64
 6   OBS_VALUE    573552 non-null  int64
dtypes: int64(3), str(4)
memory usage: 30.6 MB
None
               DATAFLOW FREQ  REF_AREA DATA_TYPE RESULT  TIME_PERIOD  \
10861

In [13]:
# with the following lines of code we can check 
# the unique content of every column to understand what they mean
print(istat_df['DATAFLOW'].unique())
print(istat_df['FREQ'].unique())
print(istat_df['TIME_PERIOD'].unique())
print(istat_df['DATA_TYPE'].unique())
print(istat_df['RESULT'].unique())

<StringArray>
['IT1:41_983(1.0)']
Length: 1, dtype: str
<StringArray>
['A']
Length: 1, dtype: str
[2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 2011 2012 2013 2014
 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024]
<StringArray>
['KILLINJ', 'ROADACC']
Length: 2, dtype: str
<StringArray>
['F', 'M', '9']
Length: 3, dtype: str


From the output we can notice that it's necessary to investigate further on the meaning of each variable

After doing some online research at the following links:

https://esploradati.istat.it/SDMXWS/rest/datastructure/IT1/DCIS_INCIDMORFER_COM/
https://esploradati.istat.it/SDMXWS/rest/codelist/IT1/CL_TIPO_DATO22
https://esploradati.istat.it/SDMXWS/rest/codelist/IT1/CL_ESITO
https://esploradati.istat.it/SDMXWS/rest/conceptscheme/IT1/CROSS_DOMAIN

you can understand the meaning of each variable. First, it is clear that each row represents a single observation, whose value is identified by the “OBS_VALUE” column, which is labeled by various categorical dimensions identified by the columns: “DATA_TYPE” and “RESULT,” which have the labels KILLINJ/ROADACC and F/M/9, respectively (F stands for injured, M for fatality, and 9 is the label for “total,” i.e., the total number of accidents that occurred in a given municipality over the course of a year).
So, for example, if you have a record with a structure like this:\

REF_AREA = 001001\
DATA_TYPE = ROADACC\
RESULT = 9\
TIME_PERIOD = 2001\
OBS_VALUE = 10\

this means that in the town with code 001001, a total of 10 traffic accidents occurred in 2001

Given the structure of the data frame, it is advisable to apply a pivot transformation to create columns showing the actual number of injuries, deaths, and accidents for each town and for each year. This will facilitate the linking of the ISTAT data with the SITUAS data.

In [14]:
# pivot trasformation
istat_tmp_df = istat_df.pivot_table(index = ['REF_AREA','TIME_PERIOD'], columns = ['DATA_TYPE','RESULT'], values = 'OBS_VALUE').reset_index()

# Rename columns to more meaningful names
istat_tmp_df.columns = ['Ref_Area','Time_Period','N_Injured','N_Dead','N_Accidents']

# Insert "Dataflow" and "Frequency" columns lost in the pivoting operation
istat_tmp_df.insert(0, 'Dataflow', istat_df['DATAFLOW'])
istat_tmp_df.insert(1, 'Frequency', istat_df['FREQ'])
istat_df = istat_tmp_df

# Convert numeric columns to int
istat_df['N_Injured'] = istat_df['N_Injured'].astype(int)
istat_df['N_Dead'] = istat_df['N_Dead'].astype(int)
istat_df['N_Accidents'] = istat_df['N_Accidents'].astype(int)
istat_df.head(10)

,Dataflow,Frequency,Ref_Area,Time_Period,N_Injured,N_Dead,N_Accidents
0,IT1:41_983(1.0),A,1001,2001,10,0,5
1,IT1:41_983(1.0),A,1001,2002,10,0,5
2,IT1:41_983(1.0),A,1001,2003,7,0,4
3,IT1:41_983(1.0),A,1001,2004,13,0,9
4,IT1:41_983(1.0),A,1001,2005,2,0,2
5,IT1:41_983(1.0),A,1001,2006,1,0,1
6,IT1:41_983(1.0),A,1001,2007,12,0,8
7,IT1:41_983(1.0),A,1001,2008,10,0,5
8,IT1:41_983(1.0),A,1001,2009,7,0,4
9,IT1:41_983(1.0),A,1001,2010,14,0,7


In [15]:
# after the pivoting of the table, we can check if there are any missing values
print(istat_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 191184 entries, 0 to 191183
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   Dataflow     191184 non-null  str  
 1   Frequency    191184 non-null  str  
 2   Ref_Area     191184 non-null  int64
 3   Time_Period  191184 non-null  int64
 4   N_Injured    191184 non-null  int64
 5   N_Dead       191184 non-null  int64
 6   N_Accidents  191184 non-null  int64
dtypes: int64(5), str(2)
memory usage: 10.2 MB
None


There are no missing values left. We can save the dataframe in a csv file

In [16]:
# Save istat dataframe to csv file in data clean folder
istat_df.to_csv("data/clean/istat_df.csv", index=False)

Now, we can join the dataframes (situas and istat) into a bigger and enriched dataframe containing information about car accidents that happened in each town in Italy each year from 2001 to 2024 (istat data). Moreover, this dataframe will have information about towns, such as their population and their surface area (situas data)

In [17]:
merged_df = pd.merge(istat_df, situas_df, left_on = ['Ref_Area','Time_Period'], right_on = ['Codice Comune (alfanumerico)','Anno'], how = 'inner')
print(merged_df.info())
print(merged_df.sample(10))

<class 'pandas.DataFrame'>
RangeIndex: 178209 entries, 0 to 178208
Data columns (total 23 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   Dataflow                        178209 non-null  str    
 1   Frequency                       178209 non-null  str    
 2   Ref_Area                        178209 non-null  int64  
 3   Time_Period                     178209 non-null  int64  
 4   N_Injured                       178209 non-null  int64  
 5   N_Dead                          178209 non-null  int64  
 6   N_Accidents                     178209 non-null  int64  
 7   Codice Ripartizione geografica  178209 non-null  int64  
 8   Codice Regione                  178209 non-null  int64  
 9   Codice Provincia/Uts            178209 non-null  int64  
 10  Codice Comune (alfanumerico)    178209 non-null  int64  
 11  Codice Comune (numerico)        178209 non-null  int64  
 12  Comune                     

From merged_df.info() we notice some errors that has to be handled:
1. "Comune" column has some null values. We can check why there are these null and decide to keep them as is, drop or fill them.
2. "Superficie (Kmq)" datatype is string and not int or float. We have to convert it in order to create other columns useful for the analysis.
3. There are redondant columns such as "Ref_Area" - "Codice Comune (alfanumerico)" - "Codice Comune (numerico)" and "Time Period" - "Anno". We can keep just one for year and one for town code.

In [23]:
# check towns code of towns that have null values in the "Comune" column after the merge operation.
merged_df[merged_df['Comune'].isnull()]

,Dataflow,Frequency,Ref_Area,Time_Period,N_Injured,N_Dead,N_Accidents,Codice Ripartizione geografica,Codice Regione,Codice Provincia/Uts,...,Sigla automobilistica,Capoluogo di Provincia/Uts,Capoluogo di Regione,Popolazione legale,Anno Censimento,Superficie (Kmq),Anno (Superficie),Popolazione residente,Anno (Popolazione residente),Anno
3892,IT1:41_983(1.0),A,1168,2001,23,0,13,1,1,1,...,TO,0,0,7761.0,2001,"24,6422",2001,7749,2001,2001
3893,IT1:41_983(1.0),A,1168,2002,46,1,20,1,1,1,...,TO,0,0,7761.0,2001,"24,6422",2002,7777,2002,2002
3894,IT1:41_983(1.0),A,1168,2003,25,4,16,1,1,1,...,TO,0,0,7761.0,2001,"24,6422",2003,7822,2003,2003
3895,IT1:41_983(1.0),A,1168,2004,29,0,15,1,1,1,...,TO,0,0,7761.0,2001,"24,6422",2004,7838,2004,2004
3896,IT1:41_983(1.0),A,1168,2005,20,3,14,1,1,1,...,TO,0,0,7761.0,2001,"24,6422",2005,7815,2005,2005
3897,IT1:41_983(1.0),A,1168,2006,32,1,20,1,1,1,...,TO,0,0,7761.0,2001,"24,6422",2006,7798,2006,2006
3898,IT1:41_983(1.0),A,1168,2007,11,0,7,1,1,1,...,TO,0,0,7761.0,2001,"24,6422",2007,7866,2007,2007
3899,IT1:41_983(1.0),A,1168,2008,42,0,19,1,1,1,...,TO,0,0,7761.0,2001,"24,6422",2008,7873,2008,2008
3900,IT1:41_983(1.0),A,1168,2009,14,0,8,1,1,1,...,TO,0,0,7761.0,2001,"24,6422",2009,7859,2009,2009
3901,IT1:41_983(1.0),A,1168,2010,8,0,6,1,1,1,...,TO,0,0,7761.0,2001,"24,6422",2010,7986,2010,2010


From the previous cell output it can be noticed that just one town has null values in the column "Comune" in the dataset. Since that we know only province and state of this town, it's not possibile to reveal its name. Moreover, null values are very few (24 vs ~170.000) so we can drop these records.

In [ ]:
# drop rows with null values in the "Comune" column and reset the index of the dataframe
merged_df.drop(merged_df[merged_df['Comune'].isnull()].index, inplace=True)
merged_df.reset_index(inplace=True, drop=True)
print(merged_df.info())
print(merged_df.sample(10))

<class 'pandas.DataFrame'>
RangeIndex: 178185 entries, 0 to 178184
Data columns (total 23 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   Dataflow                        178185 non-null  str    
 1   Frequency                       178185 non-null  str    
 2   Ref_Area                        178185 non-null  int64  
 3   Time_Period                     178185 non-null  int64  
 4   N_Injured                       178185 non-null  int64  
 5   N_Dead                          178185 non-null  int64  
 6   N_Accidents                     178185 non-null  int64  
 7   Codice Ripartizione geografica  178185 non-null  int64  
 8   Codice Regione                  178185 non-null  int64  
 9   Codice Provincia/Uts            178185 non-null  int64  
 10  Codice Comune (alfanumerico)    178185 non-null  int64  
 11  Codice Comune (numerico)        178185 non-null  int64  
 12  Comune                     